<a href="https://colab.research.google.com/github/Krittat/stds2024/blob/main/project_chat%E0%B8%9A%E0%B8%B4%E0%B8%94_%E0%B9%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install pythainlp

In [21]:
!pip install transformers

In [22]:
!pip install llama-index-llms-huggingface
!pip install llama-index

In [23]:
%pip install llama-index-embeddings-huggingface

In [24]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from pythainlp.tokenize import word_tokenize

from transformers import AutoTokenizer
import torch
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core import PromptTemplate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [25]:
!pip install -q gradio

In [26]:
import llama_index

In [27]:
pip install huggingface_hub==0.36.2

In [30]:
from huggingface_hub import  notebook_login
notebook_login()

In [31]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")

#System Prompt

In [32]:
from llama_index.core import PromptTemplate

template = """You are a knowledgeable and precise assistant specialized in answering questions based on academic and research-based sources, specifically regarding โรคติดต่อทางเพศสัมพันธ์ (STDs).

Instructions:
1. **Comprehension and Accuracy**: Carefully read and comprehend the provided context to ensure accuracy in your response.
2. **Truthfulness**: If the context lacks sufficient information to answer, respond with "ไม่มีข้อมูลในระบบครับตอนนี้".
3. **Contextual Relevance**: Use only information from the provided context. Avoid adding any outside information or assumptions.

Important:
- If no context is provided, respond with "ไม่มีข้อมูลในระบบครับตอนนี้".
- Ensure that answers are relevant, accurate, and concise.

\nQuestion: {question} \nContext: {context} \nAnswer (in Thai):"""
query_wrapper_prompt = PromptTemplate("<|USER|>{query_str}<|ASSISTANT|>")

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
from llama_index.core import SimpleDirectoryReader

required_exts = [".md"]

reader = SimpleDirectoryReader(
    input_dir="/content/drive/MyDrive/mdffile",
    required_exts=required_exts,
    recursive=True,
)
documents = reader.load_data()

In [35]:
documents[0]

Document(id_='4bfbb692-a7aa-4296-826a-748a55bbc7f1', embedding=None, metadata={'file_path': '/content/drive/MyDrive/mdffile/testalldoc1.md', 'file_name': 'testalldoc1.md', 'file_type': 'text/markdown', 'file_size': 61485, 'creation_date': '2026-06-05', 'last_modified_date': '2024-11-04'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='# โรคติดต่อทางเพศสัมพันธ์ (STDs)\r\n\r\n## บทนำ\r\nโรคติดต่อทางเพศสัมพันธ์ (STDs) หรือการติดเชื้อจากเพศสัมพันธ์ (STIs) เป็นการติดเชื้อที่แพร่กระจายผ่านการมีเพศสัมพันธ์เป็นหลัก โรคเหล่านี้เกิดจากเชื้อไวรัส แบคทีเรีย และปรสิต ที่สามารถแพร่จากคนสู่คนผ่านของเหลวในร่างกาย เช่น เลือด น้ำอสุจิ ของเหลวจากช่องคลอด แล

#LLM model

In [36]:
from transformers import AutoTokenizer

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
stopping_ids = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
]

In [37]:
from llama_index.llms.huggingface import HuggingFaceLLM
import torch

llm = HuggingFaceLLM(
    context_window=8192,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.6, "do_sample": False},
    system_prompt=template,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name=model_id,
    model_name=model_id,
    device_map="auto",
    stopping_ids=stopping_ids,
    tokenizer_kwargs={"max_length": 4096},
    # uncomment this if using CUDA to reduce memory usage
    model_kwargs={"torch_dtype": torch.float16}
)

Settings.llm = llm

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

#ใช้อยู่

In [38]:
text_splitter = SentenceSplitter(chunk_size=1024 ,chunk_overlap = 20 )
Settings.text_splitter = text_splitter
Settings.llm = llm
Settings.embed_model = embed_model

In [39]:
index = VectorStoreIndex.from_documents(documents ,transformations=[text_splitter])

In [40]:
prompt_template = PromptTemplate(template=template ,
                                 template_var_mappings = {"query_str" : "question" , "context_str" : "context"})

In [41]:
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5  # Top-k documents to retrieve
)

# Configure response synthesizer
response_synthesizer = get_response_synthesizer()

# Assemble query engine with retriever and response synthesizer
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

# Update query engine with custom prompt
query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": prompt_template}
)

def predict(input):
    print(f"Querying LLM with input: {input}")
    response = query_engine.query(input)
    print(f"Received response: {response}")
    return str(response)


In [42]:
thai_query = "สาเหตุของหนองในแท้"
output = predict(thai_query)
print("output: ",output)

Querying LLM with input: สาเหตุของหนองในแท้


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 976.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 949.81 MiB is free. Including non-PyTorch memory, this process has 13.63 GiB memory in use. Of the allocated memory 13.38 GiB is allocated by PyTorch, and 131.91 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [43]:
thai_query = "ฉันมีอาการรอยแตกและแดงบริเวณรอบอวัยวะเพศ คือโรคติดต่อทางเพศสัมพันธ์ใด"
output = predict(thai_query)
print("output: ",output)

Querying LLM with input: ฉันมีอาการรอยแตกและแดงบริเวณรอบอวัยวะเพศ คือโรคติดต่อทางเพศสัมพันธ์ใด


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 994.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 947.81 MiB is free. Including non-PyTorch memory, this process has 13.63 GiB memory in use. Of the allocated memory 12.55 GiB is allocated by PyTorch, and 982.22 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
thai_query = "อยากรู้เรื่องการป้องกันทั่วไป"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "โรคติดต่อทางเพศสัมพันธ์ ที่รักษาไม่หาย"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "การติดเชื้อ Chlamydia สามารถหายได้เองโดยไม่ต้องรักษาหรือไม่"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "โรคติดต่อทางเพศสัมพันธ์ (STDs) แพร่กระจายอย่างไร และวิธีการแพร่เชื้อหลักคืออะไร"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "อาการของหนองในเทียมขเพศชาย"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?"
output = predict(thai_query)
print("output: ",output)

In [ ]:
thai_query = "โรคหนองในแท้และหนองในเทียมแตกต่างกันอย่างไร?"
output = predict(thai_query)
print("output: ",output)

In [ ]:
import gradio as gr

gr.ChatInterface(predict).launch(share=True)